# 17 — Runtime Internals Lab

This notebook is a **hands-on dissection** of every component in `raavan.core.runtime`.
Unlike notebook 15 (shows what to *use*), this one shows **what's happening inside** as messages
flow through — we read internal state from slots directly, trigger edge cases, and
build intuition for why each piece exists.

---

| Section | Component | Focus |
|---|---|---|
| 1 | `AgentId` / `TopicId` | validation regex, hashability |
| 2 | `Mailbox` | capacity, backpressure, close protocol |
| 3 | `Dispatcher` | routing table state, reverse index, dispatch |
| 4 | `Supervisor` | restart budget, `_restart_times` deque |
| 5 | `CancellationToken` | `_futures` list, retroactive cancel |
| 6 | `LocalRuntime` – send | lazy creation, `_agents_started`, `_pending_responses` |
| 7 | Backpressure | small mailbox + slow handler |
| 8 | Fan-out timing | slow vs fast subscriber side-by-side |
| 9 | Mid-run registration | add new type while messages fly |
| 10 | `StreamPublisher` | emit-after-close, double-close safety |
| 11 | `stop_when_idle()` | `_active_handlers` counter |
| 12 | `BaseRuntime` registry | `_factories`, `_handlers`, `_topic_bindings` |

In [1]:
import sys, asyncio, time, uuid
sys.path.insert(0, "..")

from raavan.logger import setup_logging
setup_logging(mode="pretty")
print("ready")

ready


---
## 1. `AgentId` and `TopicId` — Identity, Validation, Hashability

Both are **frozen dataclasses**: immutable, hashable, usable as dict keys.
`__post_init__` enforces a regex — alphanumeric, hyphens, dots, underscores only.

In [2]:
from raavan.core.runtime._protocol import AgentId, TopicId

a1 = AgentId(type="worker", key="thread-001")
a2 = AgentId(type="worker", key="thread-002")
t1 = TopicId(type="sse_events", source="thread-001")
t2 = TopicId(type="sse_events", source="thread-002")

print(f"str(a1)     : {a1}")
print(f"str(t1)     : {t1}")
print(f"a1 == a2    : {a1 == a2}  (different key)")
print(f"t1 == t2    : {t1 == t2}  (different source)")

# Usable as dict keys
registry = {a1: "agent-1", a2: "agent-2", t1: "topic-1"}
print(f"dict lookup : {registry[a1]}")

# Frozen — cannot mutate
try:
    a1.type = "other"  # type: ignore
except Exception as e:
    print(f"mutate: {type(e).__name__}")

str(a1)     : worker/thread-001
str(t1)     : sse_events/thread-001
a1 == a2    : False  (different key)
t1 == t2    : False  (different source)
dict lookup : agent-1
mutate: FrozenInstanceError


In [3]:
# --- what the regex accepts and rejects ---
cases = [
    ("",                        "empty string"),
    ("has space",               "spaces"),
    ("has/slash",               "slash"),
    ("has@symbol",              "at-sign"),
    ("ok_hyphen-dot.under123",  "VALID: hyphens, dots, underscores, digits"),
]

for val, label in cases:
    try:
        AgentId(type=val, key="k")
        result = "PASS"
    except ValueError:
        result = "FAIL"
    print(f"  {result}  '{val}'  ({label})")

  FAIL  ''  (empty string)
  FAIL  'has space'  (spaces)
  FAIL  'has/slash'  (slash)
  FAIL  'has@symbol'  (at-sign)
  PASS  'ok_hyphen-dot.under123'  (VALID: hyphens, dots, underscores, digits)


---
## 2. `Mailbox` Internals — Bounded Queue with Close Protocol

Each agent instance owns exactly one `Mailbox`.  Internals:
- `_queue` — `asyncio.Queue(maxsize=capacity)`
- `_closed` — bool flag preventing new puts after close
- `_close_event` — `asyncio.Event` that unblocks `get()` waiters when closed

`put_nowait` raises `MailboxFullError` at capacity or after close.  
`get()` raises `StopAsyncIteration` when closed and drained.

In [4]:
from raavan.core.runtime._mailbox import Mailbox, MailboxFullError
from raavan.core.runtime._types import Envelope

sender_id = AgentId("test", "s")
target_id = AgentId("test", "t")

def make_env(payload, target=None):
    return Envelope(sender=sender_id, target=target or target_id, payload=payload)

mbox = Mailbox(capacity=3)
print(f"=== capacity=3 mailbox ===")
print(f"size    : {mbox.size}  |  closed: {mbox.closed}")
print(f"is_empty: {mbox.is_empty}  |  is_full: {mbox.is_full}")

mbox.put_nowait(make_env("msg1"))
mbox.put_nowait(make_env("msg2"))
mbox.put_nowait(make_env("msg3"))  # fills to capacity
print(f"\nafter 3 puts: size={mbox.size} is_full={mbox.is_full}")

# 4th triggers MailboxFullError
try:
    mbox.put_nowait(make_env("msg4"))
except MailboxFullError as e:
    print(f"4th put:  MailboxFullError: {e}")

=== capacity=3 mailbox ===
size    : 0  |  closed: False
is_empty: True  |  is_full: False

after 3 puts: size=3 is_full=True
4th put:  MailboxFullError: mailbox at capacity (3)


In [5]:
async def demo_mailbox_drain():
    mbox = Mailbox(capacity=5)
    for i in range(3):
        mbox.put_nowait(make_env(f"payload-{i}"))

    print(f"size before drain : {mbox.size}")

    env = await mbox.get(timeout=1.0)   # dequeue one
    print(f"got               : {env.payload}")
    print(f"size after 1 get  : {mbox.size}")

    # Close — sets _closed + fires _close_event
    mbox.close()
    print(f"\nclosed            : {mbox.closed}")
    print(f"_close_event set  : {mbox._close_event.is_set()}")

    # put after close → MailboxFullError immediately
    try:
        mbox.put_nowait(make_env("late"))
    except MailboxFullError as e:
        print(f"put after close   : MailboxFullError: {e}")

    # get drains remaining items, then raises StopAsyncIteration
    env2 = await mbox.get(timeout=1.0)   # payload-1
    env3 = await mbox.get(timeout=1.0)   # payload-2
    print(f"drained           : {env2.payload}, {env3.payload}")
    try:
        await mbox.get(timeout=1.0)      # queue drained + closed
    except StopAsyncIteration as e:
        print(f"empty+closed get  : StopAsyncIteration: {e}")

await demo_mailbox_drain()

size before drain : 3
got               : payload-0
size after 1 get  : 2

closed            : True
_close_event set  : True
put after close   : MailboxFullError: mailbox is closed
drained           : payload-1, payload-2
empty+closed get  : StopAsyncIteration: mailbox closed


---
## 3. `Dispatcher` — Routing Table Internals

Three internal dicts:

| Dict | Key | Value |
|---|---|---|
| `_mailboxes` | `AgentId` | `Mailbox` |
| `_topic_subscribers` | `TopicId` | `list[Subscription]` |
| `_type_agents` | `agent_type: str` | `set[AgentId]` (reverse index for O(1) fan-out) |

`dispatch(envelope)` inspects `envelope.target`: `AgentId` → single mailbox, `TopicId` → fan-out.

In [6]:
from raavan.core.runtime._dispatcher import Dispatcher, AgentNotFoundError

disp = Dispatcher()

# Register 3 agents — 2 of the same type, 1 different
w1 = AgentId("worker", "w1")
w2 = AgentId("worker", "w2")
l1 = AgentId("logger", "l1")
mailboxes = {aid: Mailbox(capacity=10) for aid in (w1, w2, l1)}
for aid, mb in mailboxes.items():
    disp.register_agent(aid, mb)

print("=== routing table after 3 registrations ===")
print(f"_mailboxes keys    : {list(disp._mailboxes.keys())}")
print(f"_type_agents       : {dict(disp._type_agents)}")
print(f"  worker instances : {disp._type_agents['worker']}")
print(f"  logger instances : {disp._type_agents['logger']}")
print(f"registered_agents  : {disp.registered_agents}")

=== routing table after 3 registrations ===
_mailboxes keys    : [AgentId(type='worker', key='w1'), AgentId(type='worker', key='w2'), AgentId(type='logger', key='l1')]
_type_agents       : {'worker': {AgentId(type='worker', key='w1'), AgentId(type='worker', key='w2')}, 'logger': {AgentId(type='logger', key='l1')}}
  worker instances : {AgentId(type='worker', key='w1'), AgentId(type='worker', key='w2')}
  logger instances : {AgentId(type='logger', key='l1')}
registered_agents  : [AgentId(type='worker', key='w1'), AgentId(type='worker', key='w2'), AgentId(type='logger', key='l1')]


In [7]:
async def demo_dispatcher_dispatch():
    # Envelope carries its own target — dispatch() reads envelope.target
    env = Envelope(sender=sender_id, target=w1, payload="hello from dispatcher")
    await disp.dispatch(env)

    received = await mailboxes[w1].get(timeout=1.0)
    print(f"dispatched and received : {received.payload}")
    print(f"correlation_id match    : {received.correlation_id == env.correlation_id}")
    print(f"mailbox size after get  : {mailboxes[w1].size}")

    # Dispatch to unknown AgentId → AgentNotFoundError
    ghost_env = Envelope(sender=sender_id, target=AgentId("ghost", "x"), payload="hi")
    try:
        await disp.dispatch(ghost_env)
    except AgentNotFoundError as e:
        print(f"unknown agent: AgentNotFoundError: {e}")

await demo_dispatcher_dispatch()

dispatched and received : hello from dispatcher
correlation_id match    : True
mailbox size after get  : 0
unknown agent: AgentNotFoundError: no mailbox registered for ghost/x


In [8]:
# --- pub/sub: subscribe_to_topic() populates _topic_subscribers ---
topic = TopicId("chat", "room-1")

# Both calls return a Subscription(id, topic, agent_type)
sub_worker = disp.subscribe_to_topic(topic, "worker")
sub_logger = disp.subscribe_to_topic(topic, "logger")

print(f"=== _topic_subscribers ===")
print(f"topics registered  : {disp.topics}")
print(f"subs for topic     : {len(disp._topic_subscribers[topic])}")
print(f"  sub_worker.id    : {sub_worker.id[:8]}...")
print(f"  sub_worker.type  : {sub_worker.agent_type}")
print(f"  sub_logger.type  : {sub_logger.agent_type}")

# TopicId dispatch fan-out: both worker mailboxes receive the envelope
async def demo_topic_dispatch():
    env = Envelope(sender=sender_id, target=topic, payload="broadcast!")
    await disp.dispatch(env)
    # Both worker mailboxes (w1, w2) should have received it; logger (l1) too
    for aid, mb in mailboxes.items():
        if mb.size > 0:
            e = await mb.get(timeout=0.5)
            print(f"  {aid} received: {e.payload}")

await demo_topic_dispatch()

=== _topic_subscribers ===
topics registered  : [TopicId(type='chat', source='room-1')]
subs for topic     : 2
  sub_worker.id    : 2c4cc952...
  sub_worker.type  : worker
  sub_logger.type  : logger
  worker/w1 received: broadcast!
  worker/w2 received: broadcast!
  logger/l1 received: broadcast!


In [9]:
# --- unsubscribe by id ---
print(f"before unsubscribe : {len(disp._topic_subscribers[topic])} subs")
disp.unsubscribe(sub_worker.id)  # removes by subscription id string
print(f"after unsubscribe  : {len(disp._topic_subscribers[topic])} subs (logger remains)")

# --- unregister agent: reverse index cleans up ---
print(f"\nbefore unregister w1: worker set = {disp._type_agents.get('worker')}")
disp.unregister_agent(w1)
print(f"after  unregister w1: worker set = {disp._type_agents.get('worker')}")
disp.unregister_agent(w2)
print(f"after  unregister w2: 'worker' key gone = {'worker' not in disp._type_agents}")
print(f"logger still present: {disp._type_agents.get('logger')}")

before unsubscribe : 2 subs
after unsubscribe  : 1 subs (logger remains)

before unregister w1: worker set = {AgentId(type='worker', key='w1'), AgentId(type='worker', key='w2')}
after  unregister w1: worker set = {AgentId(type='worker', key='w2')}
after  unregister w2: 'worker' key gone = True
logger still present: {AgentId(type='logger', key='l1')}


---
## 4. `Supervisor` — Restart Budget and `_restart_times` Deque

Process:
1. `supervise(agent_id, coro_factory)` — starts task, records factory for restarts  
2. On crash: prune old timestamps from `_restart_times[agent_id]`, check budget  
3. If budget not exceeded: push new timestamp, re-spawn  
4. If budget exceeded: raise `SupervisorEscalation` (bubbles to the task)

`coro_factory` must be a **zero-arg callable** that returns a new coroutine each call.

In [ ]:
import asyncio
from raavan.core.runtime._supervisor import Supervisor, SupervisorEscalation
from raavan.core.runtime._types import RestartPolicy, AgentId

# Configure the restart policy
policy = RestartPolicy(
    max_restarts=3, 
    restart_window=60.0, 
    strategy="one_for_one"
)

# Initialize the supervisor
sup = Supervisor(restart_policy=policy)
fragile_id = AgentId("fragile", "a1")
crash_count = [0]  # list so nested fn can mutate it

def fragile_factory():
    async def _run():
        crash_count[0] += 1
        await asyncio.sleep(0.01)
        raise RuntimeError(f"deliberate crash #{crash_count[0]}")
    return _run()

# Start supervision
task = sup.supervise(fragile_id, fragile_factory)

# Wait long enough for 3 restarts then escalation
await asyncio.sleep(0.25)

# Report results
print(f"crash_count             : {crash_count[0]}")
print(f"restart_times deque     : {list(sup._restart_times.get(fragile_id, []))}")
print(f"  length                : {len(sup._restart_times.get(fragile_id, []))}")
print(f"task done?              : {task.done()}")

if task.done() and not task.cancelled():
    exc = task.exception()
    if exc:
        print(f"task exception          : {type(exc).__name__}: {exc}")

await sup.stop_all()

In [ ]:
# --- one_for_all: crashing one agent restarts ALL supervised agents ---
policy_all = RestartPolicy(max_restarts=2, restart_window=60.0, strategy="one_for_all")
sup_all = Supervisor(restart_policy=policy_all)

witness_starts = []
crasher_id = AgentId("crasher", "c1")
witness_id  = AgentId("witness", "w1")

def crasher_factory():
    async def _run():
        await asyncio.sleep(0.01)
        raise RuntimeError("I crash")
    return _run()

def witness_factory():
    async def _run():
        witness_starts.append(time.monotonic())
        await asyncio.sleep(10)   # holds until cancelled
    return _run()

t_crasher = sup_all.supervise(crasher_id, crasher_factory)
t_witness  = sup_all.supervise(witness_id,  witness_factory)

await asyncio.sleep(0.15)

print(f"witness started N times : {len(witness_starts)}  (>1 due to one_for_all)")
if len(witness_starts) >= 2:
    gap = (witness_starts[1] - witness_starts[0]) * 1000
    print(f"  gap between restarts  : {gap:.0f}ms")

await sup_all.stop_all()

---
## 5. `CancellationToken` — Linking Futures

A `CancellationToken` is a shared interrupt handle passed into `send_message`.
When `.cancel()` fires, every linked `asyncio.Future` gets cancelled.

Key properties:
- Link **before** cancel → future is cancelled when token fires  
- Link **after** cancel → future is cancelled immediately (retroactive)  
- Double `.cancel()` is safe (idempotent)

In [ ]:
from raavan.core.runtime._types import CancellationToken

loop = asyncio.get_event_loop()

# --- link before cancel ---
token = CancellationToken()
f1 = loop.create_future()
f2 = loop.create_future()
token.link_future(f1)
token.link_future(f2)
print(f"before cancel: cancelled={token.cancelled}, futures linked={len(token._futures)}")

token.cancel()
print(f"after cancel : cancelled={token.cancelled}, futures cleared={len(token._futures)}")
print(f"  f1.cancelled() : {f1.cancelled()}")
print(f"  f2.cancelled() : {f2.cancelled()}")

# --- link after cancel → immediate effect ---
token2 = CancellationToken()
token2.cancel()    # already cancelled
f3 = loop.create_future()
token2.link_future(f3)   # immediately cancels f3
print(f"\nretroactive: f3.cancelled()={f3.cancelled()}")

# --- double cancel is safe ---
token3 = CancellationToken()
token3.cancel()
token3.cancel()   # no exception
print(f"double cancel safe: {token3.cancelled}")

In [ ]:
# --- use in send_message: cancel an in-flight request ---
from raavan.core.runtime._local import LocalRuntime
from raavan.core.runtime._types import MessageContext

async def demo_cancel_in_flight():
    rt = LocalRuntime(send_timeout=5.0)
    await rt.start()

    async def slow_handler(ctx: MessageContext, payload):
        await asyncio.sleep(10)   # simulate very slow work
        return "done"

    await rt.register("slow", slow_handler)

    token = CancellationToken()

    # Fire the message and cancel shortly after
    send_task = asyncio.create_task(
        rt.send_message("go", recipient=AgentId("slow", "s1"),
                        cancellation_token=token)
    )
    await asyncio.sleep(0.05)
    token.cancel()

    try:
        await send_task
    except asyncio.CancelledError:
        print("send_message was cancelled via CancellationToken")

    await rt.stop()

await demo_cancel_in_flight()

---
## 6. `LocalRuntime` — Tracing a Message End-to-End

What happens when you call `send_message`:

```
send_message(payload, recipient=AgentId("echo", "e1"))
   │
   ├─ _ensure_agent → lazy create on first message
   │     registers mailbox in Dispatcher, starts message-loop Task
   │     marks AgentId in _agents_started
   │
   ├─ Envelope(sender, target, payload, correlation_id=uuid4)
   ├─ future = Future()  →  _pending_responses[correlation_id] = future
   ├─ dispatcher.dispatch(envelope) → mailbox.put(envelope)
   └─ await future  (unblocked by message loop when handler returns)
```

In [ ]:
rt = LocalRuntime(mailbox_capacity=20, send_timeout=5.0)
await rt.start()

call_log = []

async def inspecting_handler(ctx: MessageContext, payload):
    call_log.append({
        "agent_id"       : str(ctx.agent_id),
        "sender"         : str(ctx.sender),
        "correlation_id" : ctx.correlation_id[:8],
        "payload"        : payload,
    })
    return f"echo:{payload}"

await rt.register("echo", inspecting_handler)

echo1 = AgentId("echo", "e1")
echo2 = AgentId("echo", "e2")
notebook = AgentId("notebook", "nb")

print("=== before first send ===")
print(f"_agents_started : {rt._agents_started}")
print(f"dispatcher agents: {rt._dispatcher.registered_agents}")

In [ ]:
r1 = await rt.send_message("hello", sender=notebook, recipient=echo1)

print("=== after first send ===")
print(f"result              : {r1}")
print(f"_agents_started     : {rt._agents_started}")
print(f"dispatcher agents   : {rt._dispatcher.registered_agents}")
print(f"_pending_responses  : {rt._pending_responses}  (empty: future resolved)")
print(f"handler log         : {call_log[-1]}")

In [ ]:
# Second message: same agent instance, no re-creation
r2 = await rt.send_message("world", sender=notebook, recipient=echo1)
print(f"2nd result          : {r2}")
print(f"agents count        : {len(rt._agents_started)}  (still 1 — same instance)")
print(f"total handler calls : {len(call_log)}")

# Different key = new instance
r3 = await rt.send_message("new", sender=notebook, recipient=echo2)
print(f"\nafter 2nd instance  : agents_started={len(rt._agents_started)}  (now 2)")

await rt.stop()

---
## 7. Backpressure — Small Mailbox + Slow Handler

When the mailbox fills up, `dispatcher.dispatch()` calls `mailbox.put()` which blocks  
on `asyncio.Queue.put()` — effectively back-pressuring the sender.

Timeout fires if `put()` blocks beyond `send_timeout`.

In [ ]:
async def demo_backpressure():
    # Tiny mailbox — capacity 2 — and a handler that sleeps 100ms
    rt = LocalRuntime(mailbox_capacity=2, send_timeout=3.0)
    await rt.start()

    size_snapshots = []
    slow_id = AgentId("slow", "s1")

    async def slow_handler(ctx, payload):
        mb = rt._dispatcher.get_mailbox(slow_id)
        if mb:
            size_snapshots.append(mb.size)
        await asyncio.sleep(0.1)
        return f"done:{payload}"

    await rt.register("slow", slow_handler)

    # Fire 6 messages concurrently into a capacity=2 mailbox
    tasks = [
        asyncio.create_task(rt.send_message(i, recipient=slow_id))
        for i in range(6)
    ]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    print("results             :", results)
    print("mailbox size samples:", size_snapshots)
    print("max observed size   :", max(size_snapshots, default=0))

    await rt.stop()

await demo_backpressure()

---
## 8. Fan-out Timing — Slow vs Fast Subscribers

`publish_message` delivers to **all** subscribers.  Each subscriber gets its own  
asyncio task — a slow subscriber does NOT block a fast one.

In [ ]:
async def demo_fanout_timing():
    rt = LocalRuntime()
    await rt.start()

    fast_log, slow_log = [], []

    async def fast_sub(ctx, payload):
        fast_log.append((payload, time.perf_counter()))

    async def slow_sub(ctx, payload):
        await asyncio.sleep(0.15)
        slow_log.append((payload, time.perf_counter()))

    await rt.register("fast_sub", fast_sub)
    await rt.register("slow_sub", slow_sub)

    topic = TopicId("events", "demo")
    await rt.subscribe("fast_sub", topic)
    await rt.subscribe("slow_sub", topic)

    t0 = time.perf_counter()
    await rt.publish_message("event-1", topic=topic)
    await rt.publish_message("event-2", topic=topic)

    await asyncio.sleep(0.25)   # enough for slow_sub to finish

    total_ms = (time.perf_counter() - t0) * 1000
    print(f"total elapsed           : {total_ms:.0f}ms")
    print(f"fast_sub payloads       : {[e[0] for e in fast_log]}")
    print(f"slow_sub payloads       : {[e[0] for e in slow_log]}")
    if fast_log and slow_log:
        fast_ms = (fast_log[0][1] - t0) * 1000
        slow_ms = (slow_log[0][1] - t0) * 1000
        print(f"fast received after     : {fast_ms:.1f}ms")
        print(f"slow received after     : {slow_ms:.1f}ms  ← slow didn't block fast")

    await rt.stop()

await demo_fanout_timing()

---
## 9. Mid-Run Registration — Adding Types While Others Run

New agent types can be registered at any time.  Messages sent before registration  
of the target type raise `AgentNotFoundError` immediately — there is no buffering.

In [ ]:
from raavan.core.runtime._dispatcher import AgentNotFoundError

async def demo_mid_run_registration():
    rt = LocalRuntime()
    await rt.start()

    async def echo(ctx, payload):
        return f"got:{payload}"

    await rt.register("existing", echo)
    print(f"registered types at start  : {rt.registered_types}")

    r1 = await rt.send_message(42, recipient=AgentId("existing", "e1"))
    print(f"existing type result       : {r1}")

    # Sending to unregistered type raises before hitting the mailbox
    try:
        await rt.send_message("hi", recipient=AgentId("new_type", "n1"))
    except AgentNotFoundError as e:
        print(f"before register error      : AgentNotFoundError: {e}")

    # Register mid-run — now it works
    await rt.register("new_type", echo)
    print(f"registered types after add : {rt.registered_types}")

    r2 = await rt.send_message("hi", recipient=AgentId("new_type", "n1"))
    print(f"after registration result  : {r2}")

    await rt.stop()

await demo_mid_run_registration()

---
## 10. `StreamPublisher` — Close Safety and `StreamDone` Sentinel

`StreamPublisher` wraps `publish_message` with a close protocol:
- `emit(payload)` — publishes to the topic, raises if already closed
- `close()` — sends a `StreamDone` sentinel then marks `_closed = True`
- Double `close()` — silently ignored
- Concurrent emits serialized through an `asyncio.Lock`

In [ ]:
from raavan.core.runtime._stream import StreamPublisher
from raavan.core.runtime._types import StreamDone

async def demo_stream_publisher():
    rt = LocalRuntime()
    await rt.start()

    received = []

    async def consumer(ctx, payload):
        received.append(payload)

    await rt.register("consumer", consumer)
    topic = TopicId("stream", "demo")
    await rt.subscribe("consumer", topic)

    pub = StreamPublisher(rt, topic, sender=AgentId("producer", "p1"))
    print(f"_closed before emit : {pub._closed}")

    await pub.emit("chunk-1")
    await pub.emit("chunk-2")
    await pub.emit("chunk-3")

    await pub.close()                    # sends StreamDone
    print(f"_closed after close : {pub._closed}")

    await pub.close()                    # double close — no error
    print(f"double close safe   : True")

    try:
        await pub.emit("late")
    except RuntimeError as e:
        print(f"emit after close    : RuntimeError: {e}")

    await asyncio.sleep(0.05)            # let consumer task process

    labels = [
        type(e).__name__ if isinstance(e, StreamDone) else e
        for e in received
    ]
    print(f"\nconsumer got        : {labels}")
    print(f"last is StreamDone  : {isinstance(received[-1], StreamDone)}")

    await rt.stop()

await demo_stream_publisher()

---
## 11. `stop_when_idle()` — What "Idle" Means

`stop_when_idle()` polls until all three conditions are true:
1. All mailboxes are empty (`mbox.is_empty`)
2. `_pending_responses` is empty
3. `_active_handlers == 0`

We'll watch `_active_handlers` go up as handlers start and back to 0 as they finish.

In [ ]:
async def demo_stop_when_idle():
    rt = LocalRuntime()
    await rt.start()

    completed, active_snapshots = [], []

    async def work_handler(ctx, payload):
        active_snapshots.append(rt._active_handlers)
        await asyncio.sleep(0.04)
        completed.append(payload)
        return f"done:{payload}"

    await rt.register("worker", work_handler)

    # Dispatch 10 messages across 3 agent instances concurrently
    tasks = [
        asyncio.create_task(
            rt.send_message(i, recipient=AgentId("worker", f"w{i%3}"))
        )
        for i in range(10)
    ]

    await asyncio.sleep(0)   # one tick so mailboxes fill and handlers start
    print(f"active_handlers mid-flight : {rt._active_handlers}")

    await rt.stop_when_idle(poll_interval=0.01)
    await asyncio.gather(*tasks, return_exceptions=True)

    print(f"completed count            : {len(completed)}  (expect 10)")
    print(f"active_handlers at stop    : {rt._active_handlers}  (expect 0)")
    print(f"max concurrent active      : {max(active_snapshots, default=0)}")

await demo_stop_when_idle()

---
## 12. `BaseRuntime` Registry — `_factories`, `_handlers`, `_topic_bindings`

`BaseRuntime` (parent of `LocalRuntime`) owns three datastructures:

| Slot | Type | Purpose |
|---|---|---|
| `_factories` | `dict[str, AgentFactory]` | callable to create a new instance |
| `_handlers` | `dict[str, AgentFactory]` | same object in `LocalRuntime` |
| `_topic_bindings` | `set[(agent_type, TopicId)]` | which types receive which topics |

In a remote runtime (`GrpcRuntime`), factories and handlers can diverge:  
factory creates a gRPC stub, handler routes over the network.

In [ ]:
rt = LocalRuntime()
await rt.start()

async def my_handler(ctx, payload):
    return payload

await rt.register("agent_a", my_handler)
await rt.register("agent_b", my_handler)

print("=== BaseRuntime registry ===")
print(f"registered_types   : {rt.registered_types}")
print(f"_factories keys    : {list(rt._factories.keys())}")
print(f"_handlers keys     : {list(rt._handlers.keys())}")
print(f"same object?       : {rt._factories['agent_a'] is rt._handlers['agent_a']}")

topic = TopicId("events", "demo")
await rt.subscribe("agent_a", topic)

print(f"\n_topic_bindings    : {rt._topic_bindings}")

# Subscribing an unknown type raises ValueError
try:
    await rt.subscribe("ghost", topic)
except ValueError as e:
    print(f"unknown subscribe  : ValueError: {e}")

await rt.stop()

---
## 13. Latency Benchmark — How Fast is Local Message Passing?

How much overhead does the runtime add vs a bare `await handler(ctx, payload)`?

In [ ]:
async def benchmark():
    rt = LocalRuntime(mailbox_capacity=500, send_timeout=10.0)
    await rt.start()

    async def noop(ctx, payload):
        return payload

    await rt.register("bench", noop)
    target = AgentId("bench", "b1")

    # Warmup
    for _ in range(5):
        await rt.send_message(0, recipient=target)

    N = 200

    # Sequential
    t0 = time.perf_counter()
    for i in range(N):
        await rt.send_message(i, recipient=target)
    t1 = time.perf_counter()
    seq_us = (t1 - t0) / N * 1_000_000
    print(f"Sequential {N} msgs: {(t1-t0)*1000:.1f}ms total  |  {seq_us:.0f}µs/msg")

    # Concurrent
    t0 = time.perf_counter()
    await asyncio.gather(*[rt.send_message(i, recipient=target) for i in range(N)])
    t1 = time.perf_counter()
    conc_ms = (t1 - t0) * 1000
    print(f"Concurrent  {N} msgs: {conc_ms:.1f}ms total  |  speedup vs sequential: {seq_us*N/1000/conc_ms:.1f}x")

    await rt.stop()

await benchmark()

---
## 14. Full Integration — Multi-Hop Chain with Audit Trail

Two agents bounce a counter back and forth.  
A third agent subscribes to an audit topic and records every hop.  
Shows **point-to-point + pub/sub coexisting**.

In [ ]:
async def multi_hop_with_audit():
    # Three-stage forward chain: stage1 -> stage2 -> stage3
    # Each stage publishes to audit topic before forwarding.
    # NOTE: recursive same-key send_message deadlocks because
    # the agent loop is blocked waiting for the recursive response.
    rt = LocalRuntime(send_timeout=10.0)
    await rt.start()

    audit_log = []
    audit_topic = TopicId("audit", "chain")

    async def auditor(ctx, payload):
        audit_log.append(payload)

    async def stage1(ctx, payload):
        await rt.publish_message(f"stage1 got: {payload}", topic=audit_topic)
        r = await rt.send_message(
            f"{payload}->s1", sender=ctx.agent_id, recipient=AgentId("stage2", "s2")
        )
        return f"s1->{r}"

    async def stage2(ctx, payload):
        await rt.publish_message(f"stage2 got: {payload}", topic=audit_topic)
        r = await rt.send_message(
            f"{payload}->s2", sender=ctx.agent_id, recipient=AgentId("stage3", "s3")
        )
        return f"s2->{r}"

    async def stage3(ctx, payload):
        await rt.publish_message(f"stage3 got: {payload}", topic=audit_topic)
        return f"final:{payload}"

    await rt.register("stage1",  stage1)
    await rt.register("stage2",  stage2)
    await rt.register("stage3",  stage3)
    await rt.register("auditor", auditor)
    await rt.subscribe("auditor", audit_topic)

    result = await rt.send_message("input", recipient=AgentId("stage1", "s1"))
    await asyncio.sleep(0.05)   # let audit messages drain

    n = len(audit_log)
    print(f"final result   : {result}")
    print(f"audit trail ({n} events):")
    for entry in audit_log:
        print(f"  {entry}")

    await rt.stop()

await multi_hop_with_audit()

---
## Summary

| Component | What you observed |
|---|---|
| `AgentId`/`TopicId` | Frozen, hashable; regex rejects spaces, slashes, `@` |
| `Mailbox` | `put_nowait` raises `MailboxFullError` at capacity; `get()` raises `StopAsyncIteration` when closed and drained |
| `Dispatcher` | `_mailboxes`, `_topic_subscribers`, `_type_agents` reverse index; `dispatch(envelope)` reads `envelope.target`; `subscribe_to_topic(topic, agent_type)` returns `Subscription`; unsubscribe by `sub.id` |
| `Supervisor` | `_restart_times` is a `deque` keyed by `AgentId`; `one_for_all` cancels and restarts all agents when any crashes |
| `CancellationToken` | `_futures` list; retroactive cancel for late `link_future` calls; double cancel safe |
| `LocalRuntime` | Agents created lazily on first `send_message`; same `(type, key)` = same instance; `_pending_responses` holds in-flight futures |
| Backpressure | `send_message` blocks on `mailbox.put()` — slow handlers back-pressure senders |
| Fan-out | Each subscriber gets its own task — slow sub does not block fast sub |
| Mid-run registration | `AgentNotFoundError` before registration; works immediately after |
| `StreamPublisher` | `close()` sends `StreamDone`; emit-after-close raises; double-close is safe |
| `stop_when_idle()` | Waits for `_active_handlers == 0` AND empty mailboxes AND empty `_pending_responses` |
| `BaseRuntime` | `_factories == _handlers` in `LocalRuntime`; `_topic_bindings` tracks `(type, topic)` pairs |